# Notebook 5 · Advanced LangChain

### Control the graph, split into many agents, name the patterns

`create_agent` hides a graph. Most of the time you want it hidden. Two situations pull you underneath it: you need a path the model is not allowed to improvise, or one agent is carrying too many tools and you want to split the job. Both are below, and both run offline.

**Standalone setup.** Run the cell first.

```bash
pip install langchain langgraph langchain-aws langgraph-supervisor langgraph-swarm
```

In [1]:
%pip install langchain langgraph langchain-aws langgraph-supervisor langgraph-swarm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [langgraph-swarm]
Note: you may need to restart the kernel to use updated packages.


In [2]:
from typing import Any, List, TypedDict
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage
from langchain_core.outputs import ChatResult, ChatGeneration
from langchain.agents import create_agent
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END


class ScriptedChatModel(BaseChatModel):
    '''Deterministic, credential-free stand-in. Framework code is real; replies are scripted.'''
    responses: List[Any]
    idx: int = 0

    @property
    def _llm_type(self) -> str:
        return "scripted"

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        reply = self.responses[min(self.idx, len(self.responses) - 1)]
        object.__setattr__(self, "idx", self.idx + 1)
        return ChatResult(generations=[ChatGeneration(message=reply)])

    def bind_tools(self, tools, **kwargs):
        return self


print("setup ready")

setup ready


---
## 1. When you need the graph, not the agent

An agent lets the model decide the path. A graph lets you decide the path and lets the model fill in the steps. If a compliance rule says "always check eligibility before issuing a refund", you do not want that to depend on the model remembering. You want it wired.

```mermaid
flowchart LR
    A["create_agent model picks the path"] -.you need guarantees.-> B["StateGraph you pin the path"]
```

A `StateGraph` has three parts: a state (the data passed between steps), nodes (plain functions that update the state), and edges (what runs next, fixed or conditional). Crucially, a routing decision can be pure Python, so it is deterministic and unit-testable, with no model in the loop.

### 1a. A routing graph with testable branches

We build the extractor-then-branch shape: pull a confidence score, then route to a writer on high confidence or an ambiguity handler on low. The router is a plain function.

```mermaid
flowchart TD
    S[START] --> E[extractor: score confidence]
    E --> R{confidence >= 0.7?}
    R -->|yes| W[writer]
    R -->|no| A[ambiguity handler]
    W --> X[END]
    A --> X[END]
```

> **What runs next** define state, three node functions, and a router, then wire them and run two inputs down different branches.
> **Python construct** a `TypedDict` for state, functions returning partial state updates, `add_conditional_edges` with a plain routing function.
> **LLM concept** none in the control flow. The branch is decided by code, so it behaves the same every run. That is the point of dropping to a graph.

In [3]:
class State(TypedDict):
    text: str
    confidence: float
    result: str


def extractor(state):
    conf = 0.92 if "cancel" in state["text"].lower() else 0.45
    return {"confidence": conf}


def writer(state):
    return {"result": f"resolved (confidence {state['confidence']})"}


def ambiguity(state):
    return {"result": "confidence too low, ask one clarifying question"}


def route(state):
    return "writer" if state["confidence"] >= 0.7 else "ambiguity"


graph = StateGraph(State)
graph.add_node("extractor", extractor)
graph.add_node("writer", writer)
graph.add_node("ambiguity", ambiguity)
graph.add_edge(START, "extractor")
graph.add_conditional_edges("extractor", route, {"writer": "writer", "ambiguity": "ambiguity"})
graph.add_edge("writer", END)
graph.add_edge("ambiguity", END)
app = graph.compile()

for text in ["My flight JX48Q2 is cancelled", "something feels off, not sure"]:
    out = app.invoke({"text": text, "confidence": 0.0, "result": ""})
    print(f"{text!r:36} -> conf {out['confidence']} -> {out['result']}")

'My flight JX48Q2 is cancelled'      -> conf 0.92 -> resolved (confidence 0.92)
'something feels off, not sure'      -> conf 0.45 -> confidence too low, ask one clarifying question


> **What just happened** two inputs, two branches, zero randomness. You could write a unit test asserting that a cancellation always routes to the writer, and it would pass every time. Try guaranteeing that with a model deciding the route. This is the trade: a graph costs you flexibility and buys you certainty.

> **Skeptic's corner** so why not build everything as a graph? Because most steps do not need a guarantee, and hand-wiring every path is slow and brittle as requirements shift. Use a graph for the few transitions that must hold, and let the agent handle the rest.

---
## 2. Split one agent into many

One agent with fifteen tools gets confused, and its prompt turns into a wall of caveats. Splitting into focused agents helps when the domains are genuinely distinct. Two coordination styles:

| Style | Who is in charge | Shape |
|-------|------------------|-------|
| Supervisor | a central agent delegates and collects | hub and spoke |
| Swarm | peers hand control to each other | a network, no boss |

```mermaid
flowchart TD
    subgraph Supervisor
      SUP[supervisor] --> AG1[flight agent]
      SUP --> AG2[refund agent]
      AG1 --> SUP
      AG2 --> SUP
    end
    subgraph Swarm
      F[finder] <--> B[booker]
    end
```

### 2a. Supervisor: central delegation

The supervisor reads the request, hands it to a specialist through an auto-generated handoff tool, and folds the result back into one answer. Each worker is a normal agent with a `name`.

> **What runs next** a flight specialist agent, and a supervisor that transfers to it and reports back. Scripted, so the routing is fixed and free.
> **Python construct** `create_supervisor([...], model=...).compile(...)`, agents carrying a `name`.
> **LLM concept** delegation as a tool call. The supervisor "calls" a worker the same way an agent calls a function.

In [3]:
from langgraph_supervisor import create_supervisor


@tool
def search_flights(origin: str, dest: str) -> str:
    '''Find alternate flights between two airport codes.'''
    return "AI-506 09:40, AI-812 14:15"


flight_agent = create_agent(
    ScriptedChatModel(responses=[
        AIMessage(content="", tool_calls=[{"name": "search_flights", "args": {"origin": "BLR", "dest": "DEL"}, "id": "f", "type": "tool_call"}]),
        AIMessage(content="Options: AI-506 09:40 or AI-812 14:15."),
    ]),
    tools=[search_flights],
    system_prompt="You find flights.",
    name="flight_agent",
)

# The supervisor delegates via an auto-generated tool named transfer_to_<agent name>.
supervisor_model = ScriptedChatModel(responses=[
    AIMessage(content="", tool_calls=[{"name": "transfer_to_flight_agent", "args": {}, "id": "s", "type": "tool_call"}]),
    AIMessage(content="For JX48Q2, your rebooking options are AI-506 09:40 or AI-812 14:15."),
])

supervisor = create_supervisor(
    [flight_agent],
    model=supervisor_model,
    prompt="Route each request to the right specialist, then summarise.",
).compile(checkpointer=InMemorySaver())

result = supervisor.invoke(
    {"messages": [{"role": "user", "content": "JX48Q2 cancelled, find me options BLR to DEL"}]},
    {"configurable": {"thread_id": "sup-1"}},
)
print("final:", result["messages"][-1].content)
print("messages in the run:", len(result["messages"]))

final: For JX48Q2, your rebooking options are AI-506 09:40 or AI-812 14:15.
messages in the run: 7


> **What just happened** the supervisor did not know how to search flights. It knew who did, transferred, and summarised what came back. Add a refund agent and a baggage agent the same way, and the supervisor stays thin while each specialist stays focused.

> **Gotcha** every worker needs a unique `name`, and the graph needs a checkpointer to track who is active across the handoff. Miss either and the coordination breaks in confusing ways. In production the only change is the model:

```python
# Reference (real model):
from langchain_aws import ChatBedrockConverse
model = ChatBedrockConverse(model="us.anthropic.claude-haiku-4-5-20251001-v1:0", region_name="us-east-1")
```

### 2b. Swarm: peers hand off directly

A swarm has no boss. Each agent can pass the live conversation to a peer using a handoff tool. Whoever holds control talks to the user until they hand off. This fits a natural back-and-forth better than a hub, at the cost of a wider surface to reason about.

> **What runs next** a finder that hands off to a booker. The booker finishes the job.
> **Python construct** `create_handoff_tool(agent_name=...)`, `create_swarm([...], default_active_agent=...)`.
> **LLM concept** control transfer. The handoff tool moves the active agent, it does not just return a value.

In [4]:
from langgraph_swarm import create_swarm, create_handoff_tool

to_booker = create_handoff_tool(agent_name="booker", description="Hand off to the booking agent to confirm a seat.")

finder = create_agent(
    ScriptedChatModel(responses=[
        AIMessage(content="", tool_calls=[{"name": to_booker.name, "args": {}, "id": "h", "type": "tool_call"}]),
    ]),
    tools=[to_booker],
    system_prompt="You find flights, then hand off to the booker.",
    name="finder",
)

booker = create_agent(
    ScriptedChatModel(responses=[
        AIMessage(content="Confirmed AI-506 09:40 for JX48Q2. No fee, Gold tier."),
    ]),
    tools=[],
    system_prompt="You confirm bookings.",
    name="booker",
)

swarm = create_swarm([finder, booker], default_active_agent="finder").compile(checkpointer=InMemorySaver())
result = swarm.invoke(
    {"messages": [{"role": "user", "content": "Find and book me BLR to DEL"}]},
    {"configurable": {"thread_id": "swarm-1"}},
)
print("handoff tool used:", to_booker.name)
print("final:", result["messages"][-1].content)

handoff tool used: transfer_to_booker
final: Confirmed AI-506 09:40 for JX48Q2. No fee, Gold tier.


> **What just happened** the finder handed the conversation to the booker mid-flight, and the booker closed it out. No central agent orchestrated this. Control moved sideways.

> **Skeptic's corner** supervisor or swarm? Default to supervisor. Central control is easier to reason about, log, and debug. Reach for a swarm only when the interaction is genuinely a peer conversation that a hub would make clumsy. More flexibility here means more ways to fail.

---
## 3. Five patterns that cover most systems

Most agent architectures are combinations of five shapes, from Anthropic's building-blocks taxonomy. Naming them makes design conversations faster. Two of them you already built in this notebook.

| Pattern | Shape | You build it with |
|---------|-------|-------------------|
| Prompt chaining | fixed steps, each feeds the next | a graph with linear edges |
| Routing | classify, then dispatch | conditional edges (section 1a) |
| Parallelization | fan out, then aggregate | parallel branches that merge |
| Orchestrator-workers | a lead splits and synthesises | supervisor (section 2a) |
| Evaluator-optimizer | generate, critique, retry | a loop between two nodes |

```mermaid
flowchart LR
    subgraph Chaining
      C1[A] --> C2[B] --> C3[C]
    end
    subgraph Routing
      R0[classify] --> R1[handler X]
      R0 --> R2[handler Y]
    end
    subgraph Evaluator-optimizer
      G[generate] --> J{good enough?}
      J -->|no| G
      J -->|yes| DONE[ship]
    end
```

The line between a workflow and an agent is who controls the path. Workflows pin it in code. Agents let the model choose. Real systems mix both: pin the steps that must hold, let the model handle the open-ended middle.

> **Skeptic's corner** do not reach for orchestrator-workers because it sounds sophisticated. A single agent with three good tools beats a five-agent committee for most tasks, and it is far easier to debug. Add coordination when one agent measurably struggles, not before.

---
## 4. When it breaks, you need to see inside

Multi-agent systems fail in ways a print statement cannot explain: a handoff loop, a worker called twice, a tool returning junk three hops deep. Tracing is not optional at this level.

LangSmith records every step of every run: the messages in and out of each node, each tool call and result, timings, and token counts. You set two environment variables and traces start flowing, no code change to your agent.

```bash
# Reference: enable tracing for any LangChain or LangGraph run.
export LANGSMITH_TRACING=true
export LANGSMITH_API_KEY=your_key
```

> **Gotcha** the hardest multi-agent bugs are invisible from the final answer alone, because the answer looks plausible while the path that produced it was wrong. A trace shows you the path. Turn it on before you need it, not after a production incident.

---
## What you can now do

- Build a `StateGraph` with deterministic, unit-testable routing.
- Choose between pinning a path in a graph and letting an agent decide.
- Split work across specialists with a supervisor or a swarm, and say which fits.
- Name the five workflow patterns and map each to how you would build it.
- Turn on tracing so multi-agent failures are visible instead of mysterious.

**Next, Notebook 6.** The same operations in LangChain and in Strands, side by side, so you can move between the two frameworks and pick the right one for a given job.

> **Skeptic's corner to carry forward** complexity is a cost you pay in debugging time forever, not just today. Every graph node and every extra agent is one more thing that can break at 2am. Add structure only when a simpler design has actually failed you.